# Ewaluacja RAGa

Jednym z narzędzi do ewaluacji systemów RAG i LLMów w ogólności jest biblioteka `deepeval`

https://docs.confident-ai.com/docs/getting-started

## Quickstart

Aby dokonać ewaluacji działania systemu opartego o LLM, np. RAG używamy klasy `LLMTestCase`.

Test case'y bazują na następujących parametrach:
- `input` (prompt użytkownika)
- `actual_output` (rzeczywista odpowiedź modelu)
- `expected_output` (oczekiwana odpowiedź modelu)
- `retrieval_context` (kontekst pozyskany z bazy wektorowej)

Parametrów tych jest więcej, ale w kontekście metryk dla RAGa będziemy używać wyłącznie powyższych.

> Przykłady korzystają z dziewięciu zdań o Andrzeju, Bartku i Czarku używanych we wcześniejszych notebookach. Odpowiedzi są celowo parafrazami dokumentów źródłowych, dzięki czemu metryki sprawdzają zgodność znaczenia, a nie tylko podobieństwo dosłownego brzmienia.

In [ ]:
from dotenv import load_dotenv

from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig, DisplayConfig

In [ ]:
load_dotenv()

Dokumentacja `LLMTestCase`: https://docs.confident-ai.com/docs/evaluation-test-cases

In [ ]:
test_case = LLMTestCase(
    input="Jakie zwierzę posiada Andrzej?",
    actual_output="Andrzej jest właścicielem psa.",
    expected_output="Andrzej posiada psa.",
    retrieval_context=["Andrzej ma psa."]
)

In [ ]:
answer_relevancy_metric = AnswerRelevancyMetric(threshold=0.5, model="gpt-4o-mini")
answer_relevancy_metric

In [ ]:
import os
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "100"

In [ ]:
result = evaluate([test_case], [answer_relevancy_metric])

In [ ]:
result

In [ ]:
result.test_results

In [ ]:
result.test_results[0].metrics_data[0]

In [ ]:
result.test_results[0].metrics_data[0].score

## Metryki

W bibliotece `deepeval` występuje 5 metryk, które mogą być pomocne do zastosowań związanych z RAG:

**retrieval**
- `ContextualPrecisionMetric`
- `ContextualRecallMetric`
- `ContextualRelevancyMetric`

**generation**
- `AnswerRelevancyMetric`
- `FaithfulnessMetric`


Oprócz tego istnieje również możliwość tworzenia customowych metryk za pomocą klasy `GEval`.

### Retrieval

#### `ContextualPrecisionMetric`

Ocenia, czy reranker w retrieverze umieszcza bardziej istotne fragmenty w kontekście wyszukiwania wyżej niż fragmenty mniej istotne.

Innymi słowy sprawdzamy, czy kolejność pozyskanych dokumentów jest optymalna w kontekście zadanego pytania.

https://docs.confident-ai.com/docs/metrics-contextual-precision

---

**Zestawiamy:** kolejność elementów w `retrieval_context` względem `model_input`

In [ ]:
from deepeval.metrics import ContextualPrecisionMetric


model_input = "Która osoba porusza się autem?"

actual_output = "Nie ma znaczenia"
expected_output = "Nie ma znaczenia"

# trafny fragment na pierwszym miejscu
retrieval_context_1 = ["Andrzej jeździ samochodem.",
                       "Andrzej ma psa."]

# trafny fragment na drugim miejscu
retrieval_context_2 = ["Andrzej ma psa.",
                       "Andrzej jeździ samochodem."]

# brak trafnego fragmentu
retrieval_context_3 = ["Bartek jeździ rowerem.",
                       "Andrzej lubi pływać."]


metric = ContextualPrecisionMetric(
    threshold=0.7,
    model="gpt-4o-mini"
)

test_cases = []
for retrieval_context in [retrieval_context_1, retrieval_context_2, retrieval_context_3]:
    test_case = LLMTestCase(
        input=model_input,
        actual_output=actual_output,
        expected_output=expected_output,
        retrieval_context=retrieval_context
    )

    test_cases.append(test_case)

Aby policzyć wartość metryki należy wywołać funkcję `evaluate()` przekazując do niej m.in. poniższe argumenty:
- listę przypadków testowych – może być to lista jednoelementowa
- listę metryk – również może być jednoelementowa
- `print_results` (opcjonalnie) – do kontroli *verbosity*
- `run_async` (opcjonalnie) – wywołanie asynchroniczne przyspieszy działanie kodu, ale nie zapewnia nam określonej kolejności wyników

In [ ]:
result = evaluate(
    test_cases, [metric],
    async_config=AsyncConfig(run_async=True),
    display_config=DisplayConfig(print_results=True)
)

In [ ]:
print("Model input:", result.test_results[0].input, "\n=================================================================\n")

for test_result in result.test_results:
    metric_data = test_result.metrics_data[0]
    
    print(f"score: {metric_data.score}", f"threshold: {metric_data.threshold}", f"success: {metric_data.success}",
          f"------------\nContext:\n{test_result.retrieval_context}\n------------", f"Reason:\n{metric_data.reason}", sep="\n", end="\n\n======\n\n")

#### `ContextualRecallMetric`

Ocenia, czy retriever potrafi poprawnie dobrać właściwe informacje do tego co jest w oczekiwanej odpowiedzi.

https://docs.confident-ai.com/docs/metrics-contextual-recall

---

**Zestawiamy:** `retrieval_context` oraz `expected_output`

In [ ]:
from deepeval.metrics import ContextualRecallMetric


model_input = "Nie ma znaczenia"
actual_output = "Nie ma znaczenia"
expected_output = "Andrzej podróżuje samochodem, Bartek rowerem, natomiast Czarek tramwajem"

# kontekst zawiera całą oczekiwaną informację (plus jeden zbędny fragment)
retrieval_context_1 = [
    "Andrzej jeździ samochodem.",
    "Bartek jeździ rowerem.",
    "Czarek jeździ tramwajem.",
    "Andrzej ma psa."
]

# kontekst zawiera tylko część oczekiwanej informacji
retrieval_context_2 = ["Andrzej jeździ samochodem."]

# kontekst nie zawiera oczekiwanej informacji
retrieval_context_3 = ["Bartek ma kota.", "Czarek lubi grać w szachy."]


metric = ContextualRecallMetric(
    threshold=0.7,
    model="gpt-4o-mini"
)


test_cases = []
for retrieval_context in [retrieval_context_1, retrieval_context_2, retrieval_context_3]:
    test_case = LLMTestCase(
        input=model_input,
        actual_output=actual_output,
        expected_output=expected_output,
        retrieval_context=retrieval_context
    )

    test_cases.append(test_case)

In [ ]:
result = evaluate(test_cases, [metric],
                 async_config=AsyncConfig(run_async=False),
                 display_config=DisplayConfig(print_results=False))

In [ ]:
print("Expected output:", result.test_results[0].expected_output, "\n=================================================================\n")

for test_result in result.test_results:
    metric_data = test_result.metrics_data[0]
    
    print(f"score: {metric_data.score}", f"threshold: {metric_data.threshold}", f"success: {metric_data.success}",
          f"------------\nContext:\n{test_result.retrieval_context}\n------------", f"Reason:\n{metric_data.reason}", sep="\n", end="\n\n======\n\n")

#### `ContextualRelevancyMetric`

Ocenia jaki odsetek fragmentów zwróconych przez retriever dotyczy tego, czego dotyczy prompt wejściowy.

https://docs.confident-ai.com/docs/metrics-contextual-relevancy

---

**Zestawiamy:** proporcję relewantnych tekstów w `retrieval_context` w odniesieniu do `model_input`

In [ ]:
from deepeval.metrics import ContextualRelevancyMetric


model_input = "Jakie hobby ma Czarek?"
actual_output = "Nie ma znaczenia"
expected_output = "Nie ma znaczenia"

# połowa fragmentów relewantna
retrieval_context_1 = ["Czarek lubi grać w szachy.", "Bartek ma kota."]

# wszystkie fragmenty relewantne
retrieval_context_2 = ["Czarek lubi grać w szachy."]

# żaden fragment nie jest relewantny
retrieval_context_3 = ["Bartek lubi biegać.", "Andrzej jeździ samochodem."]


metric = ContextualRelevancyMetric(
    threshold=0.7,
    model="gpt-4o-mini"
)



test_cases = []
for retrieval_context in [retrieval_context_1, retrieval_context_2, retrieval_context_3]:
    test_case = LLMTestCase(
        input=model_input,
        actual_output=actual_output,
        expected_output=expected_output,
        retrieval_context=retrieval_context
    )

    test_cases.append(test_case)

In [ ]:
result = evaluate(test_cases, [metric],
                  async_config=AsyncConfig(run_async=False),
                  display_config=DisplayConfig(print_results=False))

In [ ]:
print("Model input:", result.test_results[0].input, "\n=================================================================\n")

for test_result in result.test_results:
    metric_data = test_result.metrics_data[0]
    
    print(f"score: {metric_data.score}", f"threshold: {metric_data.threshold}", f"success: {metric_data.success}",
          f"------------\nContext:\n{test_result.retrieval_context}\n------------", f"Reason:\n{metric_data.reason}", sep="\n", end="\n\n======\n\n")

### Generation

#### `AnswerRelevancyMetric`

Określa na ile odpowiedź modelu jest adekwatna w odniesieniu do inputu użytkownika.

https://docs.confident-ai.com/docs/metrics-answer-relevancy

---

**Zestawiamy:** `model_input` i `actual_output`.

In [ ]:
from deepeval.metrics import AnswerRelevancyMetric


model_input = "Jakie zwierzę ma Bartek?"
expected_output = "Nie ma znaczenia"
retrieval_context = ["Nie ma znaczenia"]

actual_output_1 = "Bartek jest właścicielem kota."
actual_output_2 = "Bartek opiekuje się kotem i chętnie biega."
actual_output_3 = "Czarek ma królika i lubi grać w szachy."

metric = AnswerRelevancyMetric(
    threshold=0.7,
    model="gpt-4o-mini"
)


test_cases = []
for actual_output in [actual_output_1, actual_output_2, actual_output_3]:
    test_case = LLMTestCase(
        input=model_input,
        actual_output=actual_output,
        expected_output=expected_output,
        retrieval_context=retrieval_context
    )

    test_cases.append(test_case)

In [ ]:
result = evaluate(test_cases, [metric],
                 async_config=AsyncConfig(run_async=False),
                 display_config=DisplayConfig(print_results=False))

In [ ]:
print("Model input:", result.test_results[0].input, "\n=================================================================\n")

for test_result in result.test_results:
    metric_data = test_result.metrics_data[0]
    
    print(f"score: {metric_data.score}", f"threshold: {metric_data.threshold}", f"success: {metric_data.success}",
          f"------------\nActual output:\n{test_result.actual_output}\n------------", f"Reason:\n{metric_data.reason}", sep="\n", end="\n\n======\n\n")

#### `FaithfulnessMetric`

Określa czy odpowiedź modelu odnosi się w do pozyskanego z bazy kotenkstu.

https://docs.confident-ai.com/docs/metrics-faithfulness

---

**Zestawiamy:** `actual_output` i `retrieval_context`.

In [ ]:
from deepeval.metrics import FaithfulnessMetric


model_input = "Nie ma znaczenia"
actual_output = "Czarek porusza się tramwajem."
expected_output = "Nie ma znaczenia"

# kontekst potwierdza odpowiedź
retrieval_context_1 = ["Czarek jeździ tramwajem."]

# kontekst nie dotyczy odpowiedzi
retrieval_context_2 = ["Czarek ma królika."]

# kontekst przeczy odpowiedzi
retrieval_context_3 = ["Czarek jeździ rowerem."]

metric = FaithfulnessMetric(
    threshold=0.7,
    model="gpt-4o-mini"
)


test_cases = []
for retrieval_context in [retrieval_context_1, retrieval_context_2, retrieval_context_3]:
    test_case = LLMTestCase(
        input=model_input,
        actual_output=actual_output,
        expected_output=expected_output,
        retrieval_context=retrieval_context
    )

    test_cases.append(test_case)

In [ ]:
result = evaluate(test_cases, [metric],
                 async_config=AsyncConfig(run_async=False),
                 display_config=DisplayConfig(print_results=False))

In [ ]:
print("Model output:", result.test_results[0].actual_output, "\n=================================================================\n")

for test_result in result.test_results:
    metric_data = test_result.metrics_data[0]
    
    print(f"score: {metric_data.score}", f"threshold: {metric_data.threshold}", f"success: {metric_data.success}",
          f"------------\nRetrieval context:\n{test_result.retrieval_context}\n------------", f"Reason:\n{metric_data.reason}", sep="\n", end="\n\n======\n\n")

### Podsumowanie metryk

| Nazwa metryki | Co jest oceniane |
|---|---|
| `ContextualPrecision` | kolejność elementów `retrieval_context` względem `input` | 
| `ContextualRecall` | zgodność `retrieval_context` z `expected_output` |
| `ContextualRelevancy` | relewantność `retrieval_context` względem `input` |
| `AnswerRelevancy` | relewantność `actual_output` względem `input` |
| `Faithfulness` | zgodność `actual_output` z `retrieval_context` |

## Customowe metryki

W `deepeval` własną metrykę można przygotować na kilka sposobów. Dwa najczęściej spotykane podejścia to skonfigurowanie `GEval` oraz utworzenie klasy dziedziczącej po `BaseMetric`. Wybór zależy przede wszystkim od tego, czy sposób oceny można opisać językiem naturalnym, czy wymaga on własnego algorytmu.

| Kryterium | `GEval` | Własna klasa dziedzicząca po `BaseMetric` |
|---|---|---|
| Sposób definiowania | Kryteria i kroki oceny zapisane w języku naturalnym | Samodzielnie zaimplementowana logika w Pythonie |
| Najlepsze zastosowanie | Ocena semantyczna lub częściowo subiektywna, np. poprawność, kompletność albo styl | Jednoznaczne reguły, własny wzór lub bardziej złożony proces oceny |
| Kontrola nad wynikiem | Sposób obliczania wyniku jest określony przez `GEval` | Pełna kontrola nad obliczeniami i zakresem wyniku |
| Powtarzalność | Wynik może się nieznacznie zmieniać, ponieważ ocenę wykonuje LLM | Może być w pełni deterministyczna; zależy od implementacji |
| Nakład pracy | Mniejszy — wystarczy zdefiniować kryteria oceny | Większy — trzeba zaimplementować i utrzymywać całą metrykę |

### `GEval`

Użyjemy `GEval` aby stworzyć własną metrykę, która porówna oczekiwaną oraz rzeczywistą odpowiedź modelu.

`GEval` prosi model oceniający o przyznanie punktów, a następnie oblicza średnią ważoną z prawdopodobieństw możliwych ocen i normalizuje wynik do zakresu 0–1. Dlatego wynik może mieć długie rozwinięcie dziesiętne. Jawne `evaluation_steps` ograniczają dowolność oceny, a `Rubric` precyzuje, co oznaczają poszczególne przedziały punktów.

In [ ]:
from deepeval.metrics import GEval
from deepeval.metrics.g_eval import Rubric
from deepeval.test_case import SingleTurnParams

In [ ]:
metric = GEval(
    name="Semantic correctness",
    evaluation_steps=[
        "Identify the key factual claims in the actual output and expected output.",
        "Compare their meaning, not their wording. Treat synonyms, paraphrases and "
        "differences in grammar, style or formality as equivalent. For example, "
        "'X has a cat' and 'X is the owner of a cat' express the same fact.",
        "Check whether the subject, relation and object agree and whether the actual "
        "output introduces a contradiction or omits a material fact.",
        "Assign the maximum score when both outputs convey the same facts, even if "
        "they use completely different wording."
    ],
    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT
    ],
    rubric=[
        Rubric(
            score_range=(0, 1),
            expected_outcome="The outputs are unrelated or their key facts contradict each other."
        ),
        Rubric(
            score_range=(2, 4),
            expected_outcome="There is some overlap, but the main factual claim is incorrect."
        ),
        Rubric(
            score_range=(5, 7),
            expected_outcome="The main claim is correct, but a material fact is missing or added."
        ),
        Rubric(
            score_range=(8, 9),
            expected_outcome="The outputs have the same meaning with only a minor factual imprecision."
        ),
        Rubric(
            score_range=(10, 10),
            expected_outcome="The outputs are semantically equivalent; wording and style may differ."
        )
    ],
    threshold=0.7,
    model="gpt-4o-mini"
)


model_input = "Nie ma znaczenia"
actual_output = "Bartek jest właścicielem kota."

expected_outputs = [
    "Bartek ma kota.",                     # to samo znaczenie
    "Bartek prawdopodobnie ma kota.",      # drobna różnica w pewności
    "Bartek ma zwierzę domowe.",           # informacja mniej precyzyjna
    "Bartek ma kota i psa.",               # część informacji jest zgodna
    "Kot należy do kolegi Bartka.",        # podobny temat, ale inny właściciel
    "Bartek nie ma kota.",                 # bezpośrednia sprzeczność
    "Bartek lubi biegać."                  # odpowiedź niepowiązana
]


test_cases = []
for expected_output in expected_outputs:
    test_case = LLMTestCase(
        input=model_input,
        actual_output=actual_output,
        expected_output=expected_output
    )

    test_cases.append(test_case)

In [ ]:
result = evaluate(test_cases, [metric],
                 async_config=AsyncConfig(run_async=False),
                 display_config=DisplayConfig(print_results=False))

In [ ]:
print("Actual output:", result.test_results[0].actual_output, "\n=================================================================\n")

for test_result in result.test_results:
    metric_data = test_result.metrics_data[0]
    
    print(f"score: {metric_data.score:.2f}", f"threshold: {metric_data.threshold}", f"success: {metric_data.success}",
          f"------------\nExpected output:\n{test_result.expected_output}\n------------", f"Reason:\n{metric_data.reason}", sep="\n", end="\n\n======\n\n")

## Własna metryka dziedzicząca po `BaseMetric`

W `GEval` sposób obliczania wyniku jest narzucony przez bibliotekę. Dziedzicząc po `BaseMetric`, możemy jawnie zdefiniować własny wzór oparty na zgodności twierdzeń:

$$
F_1 = \frac{2PR}{P+R}\qquad score = F_1(1-C)
$$

gdzie:
- $P$ (*semantic precision*) – jaka część twierdzeń odpowiedzi rzeczywistej znajduje potwierdzenie w odpowiedzi oczekiwanej,
- $R$ (*semantic recall*) – jaka część twierdzeń odpowiedzi oczekiwanej została uwzględniona w odpowiedzi rzeczywistej,
- $C$ – stopień sprzeczności między odpowiedziami.

Wszystkie składowe należą do zakresu 0–1. Gdy $P+R=0$, przyjmujemy $F_1=0$. Model wyodrębnia twierdzenia, ocenia ich wzajemne wsparcie i wykrywa sprzeczności. Wartości $P$, $R$, $F_1$ oraz końcowy wynik są obliczane deterministycznie w Pythonie.

---

**Elementy konieczne metryki tworzonej w ten sposób:**

- klasa dziedziczy po `BaseMetric`,
- implementuje metody `measure()` oraz `a_measure()`, przyjmujące `LLMTestCase`,
- ustawia `self.score` w zakresie 0–1,
- ustawia `self.success` na podstawie `self.threshold`,
- zwraca obliczony wynik z metod `measure()` i `a_measure()`.

In [ ]:
from pydantic import BaseModel, Field

from deepeval.metrics import BaseMetric
from deepeval.models import OpenAIModel
from deepeval.test_case import LLMTestCase, SingleTurnParams

In [ ]:
class ScoredClaim(BaseModel):
    claim: str
    score: float = Field(ge=0, le=1)


class ClaimAlignmentAssessment(BaseModel):
    actual_claims: list[ScoredClaim]
    expected_claims: list[ScoredClaim]
    contradiction: float = Field(ge=0, le=1)
    reason: str


class SemanticF1Metric(BaseMetric):
    _required_params = [
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT
    ]

    def __init__(self, threshold=0.7, model="gpt-4o-mini"):
        self.threshold = threshold
        self.evaluation_model = model
        self.model = OpenAIModel(model=model, temperature=0)
        self.include_reason = True
        self.async_mode = True
        self.error = None

    @staticmethod
    def _create_prompt(test_case):
        return f"""
Evaluate the semantic agreement between the actual output and the expected output.
First, split each output into atomic claims. Return every atomic claim exactly once:
- actual_claims: claims from the actual output. For each claim, assign a score from
  0 to 1 indicating how strongly the expected output supports that claim,
- expected_claims: claims from the expected output. For each claim, assign a score
  from 0 to 1 indicating how fully the actual output covers that claim,
- contradiction: the degree to which the outputs explicitly conflict; use 0 when
  there is no conflict and 1 for a direct contradiction of a central claim.

Use semantic meaning rather than exact wording. Treat paraphrases and synonyms as
equivalent. Differences in certainty, scope, specificity, or partial support may
receive intermediate values. Do not evaluate style, grammar, or formality.
An omitted expected claim receives score=0 only in expected_claims. An additional
unsupported actual claim receives score=0 only in actual_claims. Unrelated claims
are not contradictions: contradiction requires incompatible claims about the same matter.

General examples:
1. Expected: "The library opens at 9:00 and closes at 17:00."
   Actual: "The library opens at 9:00."
   actual_claims=[("opens at 9:00", 1)]
   expected_claims=[("opens at 9:00", 1), ("closes at 17:00", 0)]
   contradiction=0
2. Expected: "The package includes a charger."
   Actual: "The package includes a charger and headphones."
   actual_claims=[("includes a charger", 1), ("includes headphones", 0)]
   expected_claims=[("includes a charger", 1)]
   contradiction=0
3. Expected: "The device costs one hundred dollars."
   Actual: "The price of the device is $100."
   actual_claims=[("costs $100", 1)]
   expected_claims=[("costs one hundred dollars", 1)]
   contradiction=0
4. Expected: "The delivery is scheduled for Friday."
   Actual: "The delivery is scheduled for Saturday."
   actual_claims=[("scheduled for Saturday", 0)]
   expected_claims=[("scheduled for Friday", 0)]
   contradiction=1
5. Expected: "The building contains a room."
   Actual: "The building contains a conference room."
   actual_claims=[("contains a conference room", 0.5)]
   expected_claims=[("contains a room", 1)]
   contradiction=0
6. Expected: "The train may be delayed."
   Actual: "The train is delayed."
   actual_claims=[("the train is delayed", 0.5)]
   expected_claims=[("the train may be delayed", 1)]
   contradiction=0
7. Expected: "The forecast predicts rain."
   Actual: "The company was founded in 1990."
   actual_claims=[("the company was founded in 1990", 0)]
   expected_claims=[("the forecast predicts rain", 0)]
   contradiction=0

Actual output: {test_case.actual_output}
Expected output: {test_case.expected_output}
"""

    @staticmethod
    def _calculate_components(assessment):
        actual_scores = [claim.score for claim in assessment.actual_claims]
        expected_scores = [claim.score for claim in assessment.expected_claims]
        precision = sum(actual_scores) / len(actual_scores) if actual_scores else 0.0
        recall = sum(expected_scores) / len(expected_scores) if expected_scores else 0.0
        f1 = 0.0 if precision + recall == 0 else (
            2 * precision * recall / (precision + recall)
        )
        score = f1 * (1 - assessment.contradiction)
        score = max(0.0, min(1.0, score))
        return precision, recall, f1, score

    def _set_result(self, assessment):
        precision, recall, f1, self.score = self._calculate_components(assessment)
        self.success = self.score >= self.threshold
        self.reason = (
            f"{assessment.reason} "
            f"Components: P={precision:.2f}, R={recall:.2f}, "
            f"C={assessment.contradiction:.2f}, F1={f1:.2f}."
        )
        return self.score

    def measure(self, test_case, *args, **kwargs):
        try:
            assessment, cost = self.model.generate(
                self._create_prompt(test_case),
                schema=ClaimAlignmentAssessment
            )
            self.evaluation_cost = cost
            return self._set_result(assessment)
        except Exception as error:
            self.error = str(error)
            raise

    async def a_measure(self, test_case, *args, **kwargs):
        try:
            assessment, cost = await self.model.a_generate(
                self._create_prompt(test_case),
                schema=ClaimAlignmentAssessment
            )
            self.evaluation_cost = cost
            return self._set_result(assessment)
        except Exception as error:
            self.error = str(error)
            raise

    def is_successful(self):
        if self.error is not None:
            self.success = False
        else:
            self.success = self.score >= self.threshold
        return self.success

    @property
    def __name__(self):
        return "Semantic Claim F1"

In [ ]:
custom_metric = SemanticF1Metric(
    threshold=0.7,
    model="gpt-4o-mini"
)

custom_result = evaluate(
    test_cases,
    [custom_metric],
    async_config=AsyncConfig(run_async=False),
    display_config=DisplayConfig(print_results=False)
)

In [ ]:
print("Actual output:", custom_result.test_results[0].actual_output,
      "\n=================================================================\n")

for test_result in custom_result.test_results:
    metric_data = test_result.metrics_data[0]

    print(
        f"score: {metric_data.score:.2f}",
        f"threshold: {metric_data.threshold}",
        f"success: {metric_data.success}",
        f"------------\nExpected output:\n{test_result.expected_output}\n------------",
        f"Reason:\n{metric_data.reason}",
        sep="\n",
        end="\n\n======\n\n"
    )